# tp-05c 重み・学習期間・ラウンド数（LightGBM 二値）

**実行場所: ホスト PC（stg）。** tp-05b の基準 `lgb_binary`（特徴量 base+S+C1）に対して、次の工夫を 1 つずつ試す（D1・D2・H1）。

| 案 | 内容 |
|---|---|
| hl1 / hl2 / hl3 | 新しい年ほど重く（半減期 1・2・3 年。検証年の前年を重み 1） |
| min2022 | 2020〜2021 年（助走期間）を学習から外す（fold 2022 は学習が空になるため比較は 2023・2024 の行で行う） |
| r150 / r500 / r800 | ラウンド数（基準は 300） |

採否: p_win の LogLoss の差の 95% 信頼区間（レース単位の paired bootstrap）の上限が 0 未満で、p_top2・p_top3 の LogLoss が悪化しないこと。条件は tp-05b と同じ（valid 2022〜2024、3 seed）。

In [ ]:
import os, sys
from pathlib import Path


def _find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "AGENTS.md").exists() and (q / "src").is_dir():
            return q
    raise RuntimeError("リポジトリルートが見つかりません")


REPO_ROOT = _find_root()
os.chdir(REPO_ROOT)
os.environ.setdefault("POLARS_MAX_THREADS", "6")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.utils.project_env import load_project_dotenv
load_project_dotenv()

In [ ]:
import json, subprocess, gc
import numpy as np
import pandas as pd
import mlflow
import psutil

from src.pipeline.mlflow.train_utils import require_tracking_server
from src.pipeline.models import tp05_data, zoo_runner, win_models as wm
from src.pipeline.models.compute import detect_compute

WORK = REPO_ROOT / "data/local/modeling/training_pipeline"
OOF_DIR = WORK / "oof/model_oof"
OUT = WORK / "model_tune_c"
OUT.mkdir(parents=True, exist_ok=True)
THREADS = detect_compute(reserve_cores=6).threads
SEEDS = [0, 1, 2]
mlflow.set_experiment("keiba-tp-model-zoo")
require_tracking_server()
GIT_COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=REPO_ROOT).stdout.strip()
D = tp05_data.load_dataset(WORK)
df, meta, folds = D["df"], D["meta"], D["folds"]
X = df[D["FEATS"]["base+S+C1"]]
FIN = meta["finish_position"].to_numpy()
print(f"行 {len(df):,} / 特徴量 {X.shape[1]} / threads {THREADS} / 使用メモリ {psutil.Process().memory_info().rss / 1e9:.1f}GB")

In [ ]:
VARIANTS = {
    "hl1": {"halflife_years": 1.0}, "hl2": {"halflife_years": 2.0}, "hl3": {"halflife_years": 3.0},
    "min2022": {"min_train_year": 2022},
    "r150": {"n_rounds": 150}, "r500": {"n_rounds": 500}, "r800": {"n_rounds": 800},
}
RES, OOFS = [], {}
base_oof = pd.read_parquet(OOF_DIR / "lgb_binary.parquet")
for name, params in VARIANTS.items():
    with mlflow.start_run(run_name=f"tp-05c-{name}", tags={"notebook": "tp-05c", "git_commit": GIT_COMMIT, "variant": name, "base": "lgb_binary"}):
        res, oof = zoo_runner.run_oof(f"lgb_binary_{name}", "lgb_binary", X, meta, folds, SEEDS, THREADS, params=dict(params), out_dir=OOF_DIR)
        num = [c for c in res.columns if c not in ("model", "fold")]
        mlflow.log_params({k: str(v) for k, v in params.items()})
        mlflow.log_metrics({f"mean_{c}": float(res[c].mean()) for c in num})
        mlflow.log_metric("seconds", res.attrs["seconds"])
        print(f"  {name:8s} win {res['logloss_win'].mean():.5f} top2 {res['logloss_top2'].mean():.5f} top3 {res['logloss_top3'].mean():.5f} ({res.attrs['seconds']:.0f}s)")
    RES.append(res); OOFS[name] = oof
    gc.collect()
pd.concat(RES).to_csv(OUT / "fold_metrics.csv", index=False)

In [ ]:
# 基準との比較: 共通の行（variant が持つ fold）で、レース単位の paired bootstrap
key = ["race_id", "horse_id"]
fin_map = meta.set_index(key)["finish_position"]
rows = []
for name, oof in OOFS.items():
    m = base_oof.merge(oof, on=key, suffixes=("_b", "_v"))
    fin = fin_map.loc[list(zip(m["race_id"], m["horse_id"]))].to_numpy()
    out = {"variant": name, "n_rows": len(m)}
    for tgt, k in (("win", 1), ("top2", 2), ("top3", 3)):
        y = (fin <= k).astype(float)
        lb, lv = wm.per_race_loss(m["race_id"].to_numpy(), m[f"p_{tgt}_b"].to_numpy(), y), wm.per_race_loss(m["race_id"].to_numpy(), m[f"p_{tgt}_v"].to_numpy(), y)
        b = wm.paired_bootstrap_diff(lb, lv, n_boot=500)
        out[f"Δ{tgt}"], out[f"{tgt}_lo"], out[f"{tgt}_hi"] = b["diff"], b["lo"], b["hi"]
    out["採用"] = bool(out["win_hi"] < 0 and out["Δtop2"] <= 0.0002 and out["Δtop3"] <= 0.0002)
    rows.append(out)
cmp_ = pd.DataFrame(rows).set_index("variant")
display(cmp_.round(5)); cmp_.to_csv(OUT / "vs_base.csv")
R = pd.concat(RES)
print("fold 別 p_win LogLoss"); display(R.pivot(index="model", columns="fold", values="logloss_win").round(5))
best = cmp_[cmp_["採用"]].sort_values("Δwin").index.tolist()
print("採用基準を満たした案（改善の大きい順）:", best)
(OUT / "report.json").write_text(json.dumps({"git_commit": GIT_COMMIT, "adopted": best, "vs_base": cmp_.round(6).reset_index().to_dict("records")}, ensure_ascii=False, indent=1, default=str), encoding="utf-8")